# Modelado 

## scikit-learn

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import roc_auc_score
import time

In [ ]:
param_grid = {'n_estimators': [10, 50, 100], 'max_depth': [5, 10, 15]}
random_forest = RandomForestClassifier(random_state=42)

# Entrenamiento con GridSearchCV
start_train = time.time()
grid = GridSearchCV(random_forest, param_grid=param_grid, scoring='roc_auc', cv=3)
grid.fit(X_train_proc, y_train)
elapsed_train = time.time() - start_train

# Predicción
start_pred = time.time()
y_pred = grid.predict(X_test_proc)
y_prob = grid.predict_proba(X_test_proc)[:, 1]
elapsed_pred = time.time() - start_pred

# Métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("\nGridSearchCV (scikit-learn) — RandomForestClassifier")
print("Mejores parámetros:", grid.best_params_)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")
print("\nMatriz de confusión:")
print(cm)
print(f"\nTiempo de entrenamiento: {elapsed_train:.2f} segundos")
print(f"Tiempo de predicción:    {elapsed_pred:.4f} segundos")


GridSearchCV (scikit-learn) — RandomForestClassifier
Mejores parámetros: {'max_depth': 15, 'n_estimators': 100}
Accuracy:  0.8008
Precision: 0.6782
Recall:    0.0036
F1-score:  0.0073
ROC AUC:   0.7036

Matriz de confusión:
[[215257     93]
 [ 53516    196]]

Tiempo de entrenamiento: 1241.38 segundos
Tiempo de predicción:    6.5252 segundos


## PySpark

In [ ]:
# ── 15. Modelo RandomForest ──────────────────────────────────────────────────
rf = SparkRFC(featuresCol="features", labelCol="label", seed=42)

# ── 16. ParamGrid ────────────────────────────────────────────────────────────
param_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 50, 100])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .build())

# ── 17. Evaluador binario ────────────────────────────────────────────────────
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# ── 18. CrossValidator ───────────────────────────────────────────────────────
cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_auc,
    numFolds=3,
    parallelism=4,
    seed=42
)

# ── 19. Entrenamiento ────────────────────────────────────────────────────────
start_train = time.time()
cv_model = cv.fit(train_df)
elapsed_train_spark = time.time() - start_train

# ── 20. Predicción ───────────────────────────────────────────────────────────
start_pred = time.time()
predictions = cv_model.transform(test_df)
predictions.persist(StorageLevel.MEMORY_AND_DISK)
predictions.count()                          # materializa
elapsed_pred_spark = time.time() - start_pred

In [ ]:
# ── 23. Mejores hiperparámetros ───────────────────────────────────────────────
best_rf = cv_model.bestModel
best_params = {
    "numTrees": best_rf.getNumTrees,
    "maxDepth": best_rf.getOrDefault(best_rf.maxDepth)
}

# ── Matriz de confusión → métricas correctas (binario, sin MulticlassEvaluator)
cm_rows = (predictions
           .groupBy("label", "prediction")
           .count()
           .orderBy("label", "prediction"))
cm_rows.show()

# Extraer TP, TN, FP, FN como variables — solo 4 filas al driver
cm_dict = {(int(r["label"]), int(r["prediction"])): r["count"]
           for r in cm_rows.collect()}

TP = cm_dict.get((1, 1), 0)
TN = cm_dict.get((0, 0), 0)
FP = cm_dict.get((0, 1), 0)
FN = cm_dict.get((1, 0), 0)

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)  if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN)  if (TP + FN) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else 0.0)

auc = evaluator_auc.evaluate(predictions)

print("══════════════════════════════════════════════════")
print("  CrossValidator (PySpark) — RandomForestClassifier")
print("══════════════════════════════════════════════════")
print(f"  Mejores parámetros : {best_params}")
print(f"  ROC AUC            : {auc:.4f}")
print(f"  Accuracy           : {accuracy:.4f}")
print(f"  Precision          : {precision:.4f}")
print(f"  Recall             : {recall:.4f}")
print(f"  F1-score           : {f1:.4f}")
print(f"  Tiempo entrenamiento (CV) : {elapsed_train_spark:.2f} s")
print(f"  Tiempo predicción         : {elapsed_pred_spark:.4f} s")
print("══════════════════════════════════════════════════")

+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|214885|
|  0.0|       1.0|     4|
|  1.0|       0.0| 53794|
|  1.0|       1.0|    11|
+-----+----------+------+

══════════════════════════════════════════════════
  CrossValidator (PySpark) — RandomForestClassifier
══════════════════════════════════════════════════
  Mejores parámetros : {'numTrees': 50, 'maxDepth': 15}
  ROC AUC            : 0.6975
  Accuracy           : 0.7998
  Precision          : 0.7333
  Recall             : 0.0002
  F1-score           : 0.0004
  Tiempo entrenamiento (CV) : 2976.65 s
  Tiempo predicción         : 12.1876 s
══════════════════════════════════════════════════
